# 03 · QLoRA Fine-tuning
Fine-tune TinyLlama-1.1B on your career-coaching dataset using QLoRA.

⏱ Expected time on RTX 3050: **2–4 hours** (3 epochs, 7 examples).

## 3.0 · Fix bitsandbytes for CUDA 12.4 (Windows)
> Run this once — copies the CUDA 12.3 DLL and renames it for 12.4. Safe to re-run.

In [1]:
import shutil, os, glob

bnb_path = os.path.join(
    os.path.dirname(__import__("bitsandbytes").__file__)
)
print(f"bitsandbytes path: {bnb_path}")

src = os.path.join(bnb_path, "libbitsandbytes_cuda123.dll")
dst = os.path.join(bnb_path, "libbitsandbytes_cuda124.dll")

if os.path.exists(dst):
    print("✅ libbitsandbytes_cuda124.dll already exists — nothing to do.")
elif os.path.exists(src):
    shutil.copy(src, dst)
    print(f"✅ Copied cuda123 → cuda124 DLL successfully.")
else:
    print("⚠️  cuda123 DLL not found. Available DLLs:")
    for f in glob.glob(os.path.join(bnb_path, "*.dll")):
        print(f"   {os.path.basename(f)}")
    print("\nIf no cuda DLLs shown, reinstall: pip install bitsandbytes==0.43.1 --prefer-binary")


bitsandbytes path: c:\Users\Aman\AppData\Local\Programs\Python\Python311\Lib\site-packages\bitsandbytes
✅ libbitsandbytes_cuda124.dll already exists — nothing to do.


## 3.1 · Verify bitsandbytes CUDA works
> Must print ✅ before continuing. If it fails, re-run cell 3.0 and restart kernel.

In [2]:
import bitsandbytes as bnb
import torch

print(f"bitsandbytes : {bnb.__version__}")
print(f"CUDA version : {torch.version.cuda}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Test the exact function that was failing
from bitsandbytes.functional import quantize_blockwise
x = torch.randn(64, 64).cuda()
quantize_blockwise(x)
print("\n✅ bitsandbytes CUDA working correctly — ready for QLoRA!")


bitsandbytes : 0.43.1
CUDA version : 12.4
GPU          : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM         : 6.4 GB

✅ bitsandbytes CUDA working correctly — ready for QLoRA!


## 3.2 · Imports & config

In [3]:
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_MODEL  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR  = "outputs/career-coach-qlora"
DATA_TRAIN  = "data/train.jsonl"
DATA_EVAL   = "data/eval.jsonl"
MAX_SEQ_LEN = 1024

print("✅ Imports OK")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ⚠️'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


✅ Imports OK
GPU : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


## 3.3 · 4-bit quantisation config
> Cuts model VRAM from ~4.4 GB (fp16) to ~1.1 GB — essential for RTX 3050.

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,         # quantise base weights to 4-bit
    bnb_4bit_quant_type       = "nf4",        # NF4 is more accurate than int4 for LLMs
    bnb_4bit_compute_dtype    = torch.bfloat16,  # actual compute in bfloat16
    bnb_4bit_use_double_quant = True,         # quantise the quant constants too (~0.4 GB saved)
)
print("✅ BnB config ready.")


✅ BnB config ready.


## 3.4 · LoRA config
> Only ~3M parameters (~0.28%) will be trained — everything else is frozen.

In [5]:
lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 8,       # LoRA rank — higher = more capacity, more VRAM
    lora_alpha     = 16,      # scaling factor (rule of thumb: 2 × r)
    lora_dropout   = 0.05,    # regularisation — prevents overfitting on small datasets
    bias           = "none",
    target_modules = [        # attention + MLP projection layers in TinyLlama
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
print("✅ LoRA config ready.")


✅ LoRA config ready.


## 3.5 · Load tokeniser

In [6]:
print(f"Loading tokeniser from {BASE_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token   # TinyLlama has no pad token
tokenizer.padding_side = "right"               # required for SFTTrainer
print(f"✅ Vocab size: {tokenizer.vocab_size:,}")


Loading tokeniser from TinyLlama/TinyLlama-1.1B-Chat-v1.0 ...


c:\Users\Aman\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Vocab size: 32,000


## 3.6 · Load quantised base model
> Downloads ~2 GB on first run. Subsequent runs load from cache.

In [7]:
print("Loading 4-bit quantised model ...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = "auto",          # automatically places layers on GPU
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
model.config.use_cache      = False        # incompatible with gradient checkpointing
model.config.pretraining_tp = 1

# Required before attaching LoRA to a quantised model
model = prepare_model_for_kbit_training(model)

vram_used = torch.cuda.memory_reserved(0) / 1e9
print(f"✅ Model loaded. VRAM reserved: {vram_used:.1f} GB")


Loading 4-bit quantised model ...


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded. VRAM reserved: 1.4 GB


c:\Users\Aman\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aman\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## 3.7 · Attach LoRA adapters

In [8]:
model = get_peft_model(model, lora_config)

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params : {train:,}  ({100*train/total:.2f}%)")
print(f"Frozen params    : {total-train:,}")
print("✅ LoRA adapters attached.")


Trainable params : 6,307,840  (1.01%)
Frozen params    : 615,606,272
✅ LoRA adapters attached.


## 3.8 · Load dataset

In [9]:
import os
if not os.path.exists(DATA_TRAIN):
    raise FileNotFoundError(
        f"Training data not found at {DATA_TRAIN}\n"
        "Run notebook 02_prepare_data.ipynb first!"
    )

dataset = load_dataset(
    "json",
    data_files={"train": DATA_TRAIN, "eval": DATA_EVAL},
    split={"train": "train", "eval": "eval"},
)
print(f"Train : {len(dataset['train'])} examples")
print(f"Eval  : {len(dataset['eval'])} examples")


Train : 7 examples
Eval  : 1 examples


## 3.9 · Training arguments
> Effective batch size = 4 × 8 = **32**. `paged_adamw_8bit` keeps optimiser states on CPU to save VRAM.

In [11]:
NUM_EPOCHS = 3   # ← increase for better results (try 5 with more data)

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = 4,    # reduce to 2 if you get CUDA out of memory
    gradient_accumulation_steps = 8,    # effective batch = 4 × 8 = 32
    per_device_eval_batch_size  = 4,
    optim                       = "paged_adamw_8bit",  # keeps optimiser on CPU
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    weight_decay                = 0.01,
    bf16                        = True,
    fp16                        = False,
    gradient_checkpointing      = True,   # saves ~1 GB VRAM, slightly slower
    logging_steps               = 10,
    evaluation_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",  # change to "wandb" after wandb login
)
print("✅ Training args ready.")


✅ Training args ready.


## 3.10 · Create trainer & start training
> Watch the `loss` values drop over steps. A loss around **1.0–1.5** is good after 3 epochs.

In [13]:
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset["train"],
    eval_dataset       = dataset["eval"],
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    packing            = False,   # ← was True — needs 100+ examples to pack
    args               = training_args,
)

print("🚀 Starting training...")
print(f"   Epochs     : {NUM_EPOCHS}")
print(f"   Train size : {len(dataset['train'])}")
print(f"   Batch size : {training_args.per_device_train_batch_size} × {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps} (effective)")
print()
trainer.train()
print("\n✅ Training complete!")

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

🚀 Starting training...
   Epochs     : 3
   Train size : 7
   Batch size : 4 × 8 = 32 (effective)



  0%|          | 0/3 [00:00<?, ?it/s]

c:\Users\Aman\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'train_runtime': 8.0049, 'train_samples_per_second': 2.623, 'train_steps_per_second': 0.375, 'train_loss': 0.6541451613108317, 'epoch': 3.0}

✅ Training complete!


## 3.11 · Save the fine-tuned adapter

In [14]:
save_path = Path(OUTPUT_DIR) / "final-adapter"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Adapter saved to: {save_path}")
print("\nFiles saved:")
for f in sorted(save_path.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name:40s} {size:8.1f} KB")
print("\n→ Move on to 04_evaluate.ipynb")


✅ Adapter saved to: outputs\career-coach-qlora\final-adapter

Files saved:
  adapter_config.json                           0.8 KB
  adapter_model.safetensors                 24679.4 KB
  README.md                                     5.0 KB
  special_tokens_map.json                       0.5 KB
  tokenizer.json                             1799.7 KB
  tokenizer.model                             488.0 KB
  tokenizer_config.json                         1.3 KB

→ Move on to 04_evaluate.ipynb


> ✅ Training done. Move on to `04_evaluate.ipynb`